In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CARGA DE DATASETS SILVER INTERMEDIOS
# ============================================================

print("=" * 60)
print("      CARGA DE DATASETS SILVER INTERMEDIOS")
print("=" * 60)

df_siaf = pd.read_parquet('ingresos_silver_intermedio.parquet')
df_sismepre = pd.read_parquet('sismepre_silver_intermedio.parquet')
df_renamu = pd.read_parquet('renamu_silver_intermedio.parquet')

print("\n✅ Datasets cargados correctamente\n")

print(f"📊 SIAF      : {len(df_siaf):,} registros | {len(df_siaf.columns)} columnas")
print(f"📊 SISMEPRE  : {len(df_sismepre):,} registros | {len(df_sismepre.columns)} columnas")
print(f"📊 RENAMU    : {len(df_renamu):,} registros | {len(df_renamu.columns)} columnas")

      CARGA DE DATASETS SILVER INTERMEDIOS

✅ Datasets cargados correctamente

📊 SIAF      : 2,852,681 registros | 29 columnas
📊 SISMEPRE  : 31,875 registros | 28 columnas
📊 RENAMU    : 1,874 registros | 1368 columnas


In [2]:
# ============================================================
# ESTANDARIZACIÓN DE UBIGEO
# ============================================================

print("\n" + "=" * 60)
print("         ESTANDARIZACIÓN DE UBIGEO")
print("=" * 60)

# SIAF
df_siaf['UBIGEO'] = df_siaf['UBIGEO'].astype(str).str.zfill(6)

# SISMEPRE
df_sismepre['UBIGEO'] = df_sismepre['SEC_EJEC'].astype(str).str.zfill(6)

# RENAMU
df_renamu['UBIGEO'] = df_renamu['Ubigeo'].astype(str).str.zfill(6)

print("\n✅ UBIGEO estandarizado correctamente")

print(f"📍 SIAF      : {df_siaf['UBIGEO'].nunique():,} UBIGEOs únicos")
print(f"📍 SISMEPRE  : {df_sismepre['UBIGEO'].nunique():,} UBIGEOs únicos")
print(f"📍 RENAMU    : {df_renamu['UBIGEO'].nunique():,} UBIGEOs únicos")



         ESTANDARIZACIÓN DE UBIGEO

✅ UBIGEO estandarizado correctamente
📍 SIAF      : 1,895 UBIGEOs únicos
📍 SISMEPRE  : 425 UBIGEOs únicos
📍 RENAMU    : 1,874 UBIGEOs únicos


In [3]:
# ============================================================
# VALIDACIÓN DE DUPLICADOS EN CLAVES
# ============================================================

print("\n" + "=" * 60)
print("        VALIDACIÓN DE DUPLICADOS")
print("=" * 60)

dup_renamu = df_renamu.duplicated(subset=['UBIGEO']).sum()

print(f"\n✅ Duplicados RENAMU por UBIGEO : {dup_renamu}")


        VALIDACIÓN DE DUPLICADOS

✅ Duplicados RENAMU por UBIGEO : 0


In [4]:
# ============================================================
# REDUCCIÓN DE COLUMNAS RENAMU
# ============================================================

print("\n" + "=" * 60)
print("        PREPARACIÓN DE RENAMU")
print("=" * 60)

columnas_renamu = [
    'UBIGEO',
    'Departamento',
    'Provincia',
    'Distrito',
    'Tipomuni',
    'ccdd',
    'ccpp',
    'ccdi'
]

columnas_disponibles = [
    c for c in columnas_renamu
    if c in df_renamu.columns
]

df_renamu_slim = df_renamu[columnas_disponibles].copy()

print(f"\n✅ RENAMU reducido a {len(columnas_disponibles)} columnas clave")


        PREPARACIÓN DE RENAMU

✅ RENAMU reducido a 8 columnas clave


In [5]:
# ============================================================
# AGREGACIÓN DE SISMEPRE
# ============================================================

print("\n" + "=" * 60)
print("        AGREGACIÓN DE SISMEPRE")
print("=" * 60)

df_sismepre_agg = df_sismepre.groupby(
    ['UBIGEO', 'ANO_APLICACION']
).agg(
    SISMEPRE_RESPUESTAS=('RESPUESTA_ID', 'count'),
    SISMEPRE_DECIMAL_PROM=('RESPUESTA_DECIMAL', 'mean'),
    SISMEPRE_ENTERO_PROM=('RESPUESTA_ENTERO', 'mean'),
    SISMEPRE_ESTADO_MODA=('ESTADO', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
).reset_index()

print("\n✅ SISMEPRE agregado correctamente")

print(f"📊 Registros agregados : {len(df_sismepre_agg):,}")
print(f"📋 Columnas resultantes: {len(df_sismepre_agg.columns)}")



        AGREGACIÓN DE SISMEPRE

✅ SISMEPRE agregado correctamente
📊 Registros agregados : 425
📋 Columnas resultantes: 6


In [6]:
# ============================================================
# JOIN 1: SIAF + RENAMU
# ============================================================

print("\n" + "=" * 60)
print("             JOIN 1: SIAF + RENAMU")
print("=" * 60)

df_siaf_renamu = pd.merge(
    df_siaf,
    df_renamu_slim,
    on='UBIGEO',
    how='left',
    validate='many_to_one'
)

print("\n✅ JOIN SIAF + RENAMU completado")

print(f"📊 Registros finales          : {len(df_siaf_renamu):,}")
print(f"📍 Municipalidades con match  : {df_siaf_renamu['Departamento'].notna().sum():,}")
print(f"⚠️  Municipalidades sin match : {df_siaf_renamu['Departamento'].isna().sum():,}")


             JOIN 1: SIAF + RENAMU

✅ JOIN SIAF + RENAMU completado
📊 Registros finales          : 2,852,681
📍 Municipalidades con match  : 2,844,078
⚠️  Municipalidades sin match : 8,603


In [7]:
# ============================================================
# JOIN 2: (SIAF + RENAMU) + SISMEPRE
# ============================================================

print("\n" + "=" * 60)
print("      JOIN 2: + SISMEPRE")
print("=" * 60)

df_gold = pd.merge(
    df_siaf_renamu,
    df_sismepre_agg,
    left_on=['UBIGEO', 'ANO_DOC'],
    right_on=['UBIGEO', 'ANO_APLICACION'],
    how='left'
)

df_gold.drop(columns=['ANO_APLICACION'], inplace=True, errors='ignore')

print("\n✅ JOIN final completado")

print(f"📊 Registros totales     : {len(df_gold):,}")
print(f"📋 Columnas totales      : {len(df_gold.columns)}")
print(f"📍 UBIGEOs únicos        : {df_gold['UBIGEO'].nunique():,}")


      JOIN 2: + SISMEPRE

✅ JOIN final completado
📊 Registros totales     : 2,852,681
📋 Columnas totales      : 40
📍 UBIGEOs únicos        : 1,895


In [8]:
# ============================================================
# VALIDACIÓN FINAL
# ============================================================

print("\n" + "=" * 60)
print("          VALIDACIÓN FINAL DATASET GOLD")
print("=" * 60)

print(f"\n📊 Total registros : {len(df_gold):,}")
print(f"📋 Total columnas  : {len(df_gold.columns)}")

columnas_validar = [
    'UBIGEO',
    'Departamento',
    'MONTO_PIA',
    'MONTO_PIM',
    'MONTO_RECAUDADO'
]

for col in columnas_validar:

    if col in df_gold.columns:

        nulos = df_gold[col].isnull().sum()
        porcentaje = (nulos / len(df_gold)) * 100

        estado = "✅" if nulos == 0 else "⚠️"

        print(f"{estado} {col:<20}: {nulos:,} nulos ({porcentaje:.2f}%)")



          VALIDACIÓN FINAL DATASET GOLD

📊 Total registros : 2,852,681
📋 Total columnas  : 40
✅ UBIGEO              : 0 nulos (0.00%)
⚠️ Departamento        : 8,603 nulos (0.30%)
✅ MONTO_PIA           : 0 nulos (0.00%)
✅ MONTO_PIM           : 0 nulos (0.00%)
✅ MONTO_RECAUDADO     : 0 nulos (0.00%)


In [9]:
# ============================================================
# EXPORTACIÓN FINAL
# ============================================================

print("\n" + "=" * 60)
print("          EXPORTACIÓN DATASET GOLD")
print("=" * 60)

for col in df_gold.columns:

    if df_gold[col].dtype == 'object':
        df_gold[col] = df_gold[col].astype(str)

df_gold.to_parquet(
    'gold_municipalidades.parquet',
    index=False
)

print("\n✅ Dataset Gold exportado correctamente")

print(f"📁 Archivo   : gold_municipalidades.parquet")
print(f"📊 Registros : {len(df_gold):,}")
print(f"📋 Columnas  : {len(df_gold.columns)}")

print("\n✅ Proceso ETL completado correctamente")


          EXPORTACIÓN DATASET GOLD

✅ Dataset Gold exportado correctamente
📁 Archivo   : gold_municipalidades.parquet
📊 Registros : 2,852,681
📋 Columnas  : 40

✅ Proceso ETL completado correctamente
